# R03 — Train VAE

ConvVAE3D v2 (no attention), **XCT-only encoder** (`in_channels=1`), `z_channels=16`, `free_bits=0.25`, constant LR, no KL warmup.  
Config: `src/poregen/configs/r03.yaml`  
Run on an interactive HPC node with GPU(s).

In [1]:
from poregen.configs.config import load_config
from pathlib import Path
from torch.utils.data import DataLoader
import torch

from poregen.dataset.loader import PatchDataset
from poregen.models.vae import build_vae
from poregen.losses import compute_total_loss
from poregen.training import (
    seed_everything,
    select_device,
    get_autocast_dtype,
    make_scaler,
    train_loop,
)

## Config

In [2]:
CONFIG_PATH = "../src/poregen/configs/r03.yaml"
cfg = load_config(CONFIG_PATH)

# DGX Spark overrides
cfg["data"]["num_workers"] = 12

VAL_BATCHES = 60
MONTECARLO_EVERY = 100
MONTECARLO_BATCH_SIZE = 8
FINAL_FULL_EVAL = True

print(cfg)

{'model': {'name': 'v2.conv_noattn', 'in_channels': 1, 'z_channels': 16, 'base_channels': 32, 'n_blocks': 2, 'patch_size': 64}, 'loss': {'xct_loss_type': 'charbonnier', 'xct_weight': 1.0, 'mask_bce_weight': 1.0, 'mask_bce_pos_weight': 17.16, 'mask_dice_weight': 1.0, 'use_tversky': True, 'tversky_alpha': 0.5, 'tversky_beta': 0.5, 'kl_free_bits': 0.25, 'kl_warmup_steps': 0, 'kl_max_beta': 0.05}, 'training': {'seed': 42, 'lr': 0.0002, 'weight_decay': 0.01, 'total_steps': 71690, 'max_grad_norm': None, 'scheduler': 'none', 'lr_min': 2e-05, 'warmup_steps': 0, 'log_every': 1, 'eval_every': 62, 'val_batches': 100, 'test_every': 625, 'test_batches': 20, 'save_every': 1000, 'image_log_every': 62, 'montecarlo_every': 100, 'montecarlo_batch_size': 8, 'final_full_eval': True, 'sample_every': 12500, 'n_patch_samples': 8, 'compile': True, 'deterministic': False}, 'data': {'split_version': 'v2', 'batch_size': 128, 'num_workers': 12, 'pin_memory': True, 'dataset_root': 'split_v2'}}


## Setup

In [3]:
seed_everything(cfg["training"]["seed"])
device = select_device()  # or select_device(gpu_id=1)
autocast_dtype = get_autocast_dtype(device)
scaler = make_scaler(device)
print(f"Device: {device}  |  AMP dtype: {autocast_dtype}")

Device: cuda:0  |  AMP dtype: torch.bfloat16


## Data

In [4]:
DATA_ROOT = Path("../data") / cfg["data"].get("dataset_root", "split_v2")
INDEX = DATA_ROOT / "patch_index.parquet"
train_ds = PatchDataset(INDEX, DATA_ROOT, split="train")
val_ds   = PatchDataset(INDEX, DATA_ROOT, split="val")
test_ds  = PatchDataset(INDEX, DATA_ROOT, split="test")

dl_kwargs = dict(
    batch_size=cfg["data"]["batch_size"],
    num_workers=cfg["data"]["num_workers"],
    pin_memory=cfg["data"].get("pin_memory", True),
)
if cfg["data"]["num_workers"] > 0:
    dl_kwargs["persistent_workers"] = True
    dl_kwargs["prefetch_factor"] = cfg["data"].get("prefetch_factor", 4)

val_generator = torch.Generator().manual_seed(cfg["training"]["seed"] + 1)

train_loader = DataLoader(train_ds, shuffle=True, drop_last=True, **dl_kwargs)
val_loader   = DataLoader(val_ds,   shuffle=True, generator=val_generator, **dl_kwargs)
test_loader  = DataLoader(test_ds,  shuffle=False, **dl_kwargs)

train_steps_per_epoch = len(train_loader)
val_steps_per_epoch   = len(val_loader)

# ── Partial val (val/) ────────────────────────────────────────────────────
# Runs every eval_every steps over VAL_BATCHES random batches → val/ in TB.
# eval_every is chosen so we cover ~all val batches once per epoch.
cfg["training"]["val_batches"] = min(100, int(VAL_BATCHES))
val_windows_per_epoch = max(
    1,
    (val_steps_per_epoch + cfg["training"]["val_batches"] - 1)
    // cfg["training"]["val_batches"],
)
cfg["training"]["eval_every"] = max(1, train_steps_per_epoch // val_windows_per_epoch)

# ── Full val (val_full/) ──────────────────────────────────────────────────
# Runs once per epoch (every train_steps_per_epoch steps), iterates the
# entire val loader, and logs to val_full/ in TB.
FULL_VAL_EVERY = train_steps_per_epoch   # one complete epoch

# Keep image logging synchronised with partial val.
cfg["training"]["image_log_every"]       = cfg["training"]["eval_every"]
cfg["training"]["montecarlo_every"]      = int(MONTECARLO_EVERY)
cfg["training"]["montecarlo_batch_size"] = int(MONTECARLO_BATCH_SIZE)
cfg["training"]["final_full_eval"]       = bool(FINAL_FULL_EVAL)

print(f"Split root:           {DATA_ROOT}")
print(f"Train / val / test:   {len(train_ds):,} / {len(val_ds):,} / {len(test_ds):,} patches")
print(f"Train steps/epoch:    {train_steps_per_epoch:,}")
print(f"Partial val cadence:  every {cfg['training']['eval_every']} steps  ({cfg['training']['val_batches']} batches → val/)")
print(f"Full val cadence:     every {FULL_VAL_EVERY} steps = 1 epoch       (all {val_steps_per_epoch} batches → val_full/)")

Split root:           ../data/split_v2
Train / val / test:   1,822,599 / 229,195 / 222,829 patches
Train steps/epoch:    14,239
Partial val cadence:  every 474 steps  (60 batches → val/)
Full val cadence:     every 14239 steps = 1 epoch       (all 1791 batches → val_full/)


## Model + Optimizer

In [5]:
model = build_vae(
    cfg["model"]["name"],
    in_channels=cfg["model"]["in_channels"],
    z_channels=cfg["model"]["z_channels"],
    base_channels=cfg["model"]["base_channels"],
    n_blocks=cfg["model"]["n_blocks"],
    patch_size=cfg["model"]["patch_size"],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=cfg["training"]["lr"],
    weight_decay=cfg["training"]["weight_decay"],
)

scheduler = None  # constant LR throughout (scheduler: none in config)

print(f"Model: {cfg['model']['name']}  in_channels={cfg['model']['in_channels']}  z_channels={cfg['model']['z_channels']}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Model: v2.conv_noattn  in_channels=1  z_channels=16
Params: 495,842


## Loss function

In [6]:
loss_fn = lambda output, batch, step: compute_total_loss(output, batch, step, cfg)

## Run

In [ ]:
from torch.utils.tensorboard import SummaryWriter
import shutil
import importlib
import poregen.training.engine as _engine_module
importlib.reload(_engine_module)
from poregen.training.engine import train_loop

m = cfg["model"]
t = cfg["training"]
l = cfg["loss"]
d = cfg["data"]

# Run name encodes every decision that affects training dynamics.
# Format: {arch}_z{z_ch}_c{base_ch}_bs{bs}_lr{lr}_b{beta}_fb{fb}_klw{klw}_sched{sched}_clip{clip}
arch = m['name'].replace('.', '-')
EXPERIMENT_ID = "r03-"
EXP_NAME = (
    f"{EXPERIMENT_ID}"
    f"{arch}"
    f"_z{m['z_channels']}"
    f"_c{m['base_channels']}"
    f"_bs{d['batch_size']}"
    f"_lr{t['lr']:.0e}"
    f"_b{l['kl_max_beta']:.3f}"
    f"_fb{l['kl_free_bits']}"
    f"_klw{l['kl_warmup_steps']}"
    f"_sched{t['scheduler']}"
    f"_clip{'none' if t['max_grad_norm'] is None else t['max_grad_norm']}"
)
RUN_DIR = Path(f"../runs/vae/{EXP_NAME}")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print(f"Run: {EXP_NAME}")

shutil.copy(CONFIG_PATH, RUN_DIR / "config.yaml")
print(f"Config saved → {RUN_DIR / 'config.yaml'}")

tb_writer = SummaryWriter(str(RUN_DIR / "tb"))

history = train_loop(
    model,
    train_loader,
    val_loader,
    optimizer,
    scaler,
    loss_fn,
    total_steps=cfg["training"]["total_steps"],
    log_every=cfg["training"]["log_every"],
    eval_every=cfg["training"]["eval_every"],
    val_batches=cfg["training"]["val_batches"],
    test_loader=test_loader,
    test_every=cfg["training"]["test_every"],
    test_batches=cfg["training"]["test_batches"],
    save_every=cfg["training"]["save_every"],
    image_log_every=cfg["training"]["image_log_every"],
    montecarlo_every=cfg["training"]["montecarlo_every"],
    montecarlo_batch_size=cfg["training"]["montecarlo_batch_size"],
    sample_every=cfg["training"]["sample_every"],
    n_patch_samples=cfg["training"]["n_patch_samples"],
    run_dir=RUN_DIR,
    device=device,
    autocast_dtype=autocast_dtype,
    max_grad_norm=cfg["training"]["max_grad_norm"],
    scheduler=scheduler,
    tb_writer=tb_writer,
    full_val_every=FULL_VAL_EVERY,
    final_full_eval=cfg["training"]["final_full_eval"],
    compile_model=cfg["training"].get("compile", False),
)

tb_writer.close()
print(f"Done. Final loss: {history[-1]['total']:.4f}")

Run: r03-v2-conv_noattn_z16_c32_bs128_lr2e-04_b0.050_fb0.25_klw0_schednone_clipnone
Config saved → ../runs/vae/r03-v2-conv_noattn_z16_c32_bs128_lr2e-04_b0.050_fb0.25_klw0_schednone_clipnone/config.yaml


Training:   0%|          | 0/71690 [00:00<?, ?it/s]

## Quick loss plot

In [ ]:
import matplotlib.pyplot as plt

train_h = [r for r in history if r["split"] == "train"]
fig, ax = plt.subplots(1, 1, figsize=(8, 3))
ax.plot([r["step"] for r in train_h], [r["total"] for r in train_h], lw=0.6)
ax.set(xlabel="Step", ylabel="Total loss", title="R03 training loss")
plt.tight_layout()
plt.show()